# Ride Demand & Revenue Analysis - 35,000 Trips

**The brief:** a ride-hailing operator needs to know *when* to surge-price, *where* to
position premium vehicles, and *which* fleet tier to grow.

This notebook is the exploratory half of the project. The business answers are summarised
in [README.md](README.md) as **Q1-Q8**; the production queries live in
`sql_queries/02_revenue_analysis.sql`. Each section below is labelled with the README
Q-number it supports.

> *Data note: this is a simulated dataset. Fares are in INR - the data uses Indian
> payment methods including UPI.*

---

## Section 0 - Setup

In [ ]:
#Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder


import warnings
warnings.filterwarnings('ignore')





---

## Section 1 - Load & Validate

**Question before any analysis: can I trust this data?** Row count and a `head()` come
first, so every number later in the notebook is traceable to a known starting population.

In [ ]:
# LOAD DATASET
# Uses the committed cleaned extract so this notebook runs straight from a
# `git clone` with no external file needed.
df = pd.read_csv("data/Uber_Cleaned_data.csv")

print(df.head())

In [ ]:
print(df.shape)

---

## Section 2 - Cleaning & Feature Engineering

Dates and times arrive as strings, so nothing time-based works until they are parsed.
`Hour` is extracted here because **every demand finding in this project is an
hour-of-day finding** - it is the single most load-bearing engineered column.

The `dropna` below prints rows before/after, so the cleaning impact is measurable rather
than silent. Dropping rows without reporting how many is how analyses quietly lose
population.

In [ ]:
#Data Cleaning


# Convert Date to Datetime format
df['Date'] = pd.to_datetime(df['Date'])

#convert time column to datetime
df['Time'] = pd.to_datetime(df['Time'])

#Extract Hour from datetime
df['Hour'] = df['Time'].dt.hour


#convert column to numeric
df['Fare_Amount'] = pd.to_numeric(df['Fare_Amount'], errors='coerce')
df['Trip_Distance_km'] = pd.to_numeric(df['Trip_Distance_km'], errors='coerce')
df['Trip_Duration_min'] = pd.to_numeric(df['Trip_Duration_min'], errors='coerce')



In [ ]:
#Remove missing values (show before/after so the cleaning impact is measurable)
rows_before = len(df)

df.dropna(subset=['Fare_Amount',
                  'Pickup_Location',
                  'Dropoff_Location',
                  'Trip_Distance_km'], inplace=True)

rows_after = len(df)
print(f"Rows before: {rows_before:,}  |  after: {rows_after:,}  |  dropped: {rows_before - rows_after:,}")
print("Note: the source data is already complete on these key columns (0 dropped);")
print("the dropna is kept as a defensive safeguard, not because rows were removed.")

**Feature engineering - the airport flag.** Airport trips are priced differently from
city trips, so they need to be separable. This one boolean column is what makes **Q2**
(Rs 535.77 airport vs Rs 289.40 city, 1.85x) answerable.

In [ ]:
# Feaature Engineering
# Create airport Trip column
df['Is_Airport_Trip'] = (
    (df['Pickup_Location'] == 'Airport') |
    (df['Dropoff_Location'] == 'Airport')
)

print(df[['Pickup_Location','Dropoff_Location','Is_Airport_Trip']].head())

---

## Section 3 - Exploratory Data Analysis

### Q1 - When should we surge-price and add drivers?

*Metric: trip count by hour.*

**Takeaway:** demand is not spread across the day - **5-7 PM carries 31.2% of all
demand** (10,931 of 35,000 trips) in just 3 of 24 hours. Peak is 6 PM at 4,053 trips.

The SQL goes one step further (section 9, `LAG()`): the rush *ignites* at 5 PM, +138% over
4 PM - an hour before the volume peak. That is **Q7**, and it moves the incentive
trigger to 4:30 PM.

In [ ]:
#Exploratory data Analysis(EDA)
trip_hour = df['Hour'].value_counts().sort_index()

plt.figure(figsize=(8,5))
sns.lineplot(x=trip_hour.index, y=trip_hour.values)

plt.title("Trips by Hour")
plt.xlabel("Hour")
plt.ylabel("Number of Trips")
plt.show()


### Q1 (revenue view) - does the demand peak also carry the revenue?

*Metric: total fare by hour.*

**Takeaway:** yes - the revenue curve tracks the volume curve, so the evening peak is a
genuine money window, not just a busy one. If they diverged, surge pricing would need a
different target hour.

In [ ]:
#Revenue  By Hour

hourly_revenue = df.groupby('Hour')['Fare_Amount'].sum().reset_index()
plt.figure(figsize=(8,5))
sns.barplot(data=hourly_revenue, x='Hour', y='Fare_Amount')

plt.title("Revenue by Hour")
plt.show()

### Q3 support - is fare actually driven by distance?

*Metric: fare vs trip distance scatter.*

**Takeaway:** a clear positive relationship, which matters more than it looks. It is the
reason **Q4** resolves the way it does - once you know distance drives fare, comparing
raw average fares across driver-rating bands is comparing trip lengths, not drivers.

In [ ]:
#Distance Vs Fare

plt.figure(figsize=(6,4))
sns.scatterplot(
    data=df,
    x='Trip_Distance_km',
    y='Fare_Amount'
)

plt.title("Distance vs Fare")
plt.show()

### Q2 - how dependent are we on airport trips?

*Metric: average fare, airport vs city.*

**Takeaway:** airport trips are **39.6% of volume but 54.8% of revenue** (Rs 535.77 vs
Rs 289.40). This is the highest-value corridor in the business and the strongest case for
a dedicated dispatch strategy.

In [ ]:
#Airport Vs City Fare
avg_fares = df.groupby('Is_Airport_Trip')['Fare_Amount'].mean()

print("Average Airport Fare:", avg_fares.get(True))
print("Average City Fare:", avg_fares.get(False))

### The confounder check - before trusting any single-variable comparison

*Metric: correlation matrix across all numeric fields.*

**Takeaway:** rating correlates **-0.13** with trip distance and only **-0.09** with
fare. Longer trips get rated worse. That is the whole of **Q4**: the apparent
"low-rated drivers earn more" effect is trip length wearing a disguise. Fare-per-km is
flat at Rs 22.06-22.41 across every rating band.

Running this matrix *before* writing recommendations is what stopped a wrong one from
shipping.

In [ ]:
#Correlation Heatmap

numeric_cols = df.select_dtypes(include=np.number)
corr = numeric_cols.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

---

## Section 3b - Business Insight Summary

The cells below restate the findings as plain numbers, so they can be lifted straight
into a stakeholder summary without re-reading the charts.

In [ ]:
# Bussiness Insight

# Peak Hour Demand
peak_hour = df['Hour'].value_counts().idxmax()
print(f"Peak trip demand occurs at hour: {peak_hour}")

# Airport vs City Fare
avg_fares = df.groupby('Is_Airport_Trip')['Fare_Amount'].mean()

airport_fare = avg_fares.get(True, 0)
city_fare = avg_fares.get(False, 0)

print(f"Average Airport Trip Fare: ₹{airport_fare:.2f}")
print(f"Average City Trip Fare: ₹{city_fare:.2f}")


**Q6 & Q8 - the negative findings.** Weekend fares differ by **0.9%** (Rs 389.34 vs
Rs 386.05) and the best month beats the worst by only **11%**. Neither justifies a pricing
tier or a seasonal capacity plan.

**Reporting these matters.** An earlier draft of this project recommended a weekend
surcharge; the data does not support one, so it was removed. Knowing which
recommendations the data *cannot* carry is part of the job.

In [ ]:
# Distance Impact on Fare
corr_distance_fare = df[['Trip_Distance_km','Fare_Amount']].corr().iloc[0,1]
print(f"Correlation between Distance and Fare: {corr_distance_fare:.2f}")

# Weekend Trip Pattern
weekend_trips = df['Is_Weekend'].value_counts()

print("\nWeekend vs Weekday Trips:")
print(weekend_trips)

## Section 4 - Fare Prediction Model (supporting, not the headline)

**Business question:** can we estimate a fare before the trip starts, for upfront pricing?

This is a supporting model, not the main deliverable. The business findings above come
from the SQL and EDA; this section checks whether the fare structure is learnable at all.

In [ ]:
#Machine Learning-Fare rediction

features = [
'Trip_Distance_km','Trip_Duration_min','Vehicle_Type','Pickup_Location','Dropoff_Location','Hour',
'Is_Weekend'
]

ml_df = df[features + ['Fare_Amount']].copy()

In [ ]:
# Encode categorical Variable

le = LabelEncoder()
categorical_cols = ['Vehicle_Type','Pickup_Location','Dropoff_Location'
]

for col in categorical_cols:
    ml_df[col] = le.fit_transform(ml_df[col])

In [ ]:
#train Test Split

X = ml_df.drop('Fare_Amount', axis=1)
y = ml_df['Fare_Amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#Train Model

model = RandomForestRegressor(n_estimators=100,random_state=42)
model.fit(X_train, y_train)

In [ ]:
#Model Evaluation
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

**Feature importance - a sanity check on the model.**

If distance did not dominate here, the model would be learning noise. It confirms the
same structure the EDA showed, which is the point: the model agrees with the analysis
rather than replacing it.

In [ ]:
# Feature Importance

feature_importance = pd.DataFrame({
'Feature': X.columns,
'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

plt.figure(figsize=(8,4))
sns.barplot(x='Importance', y='Feature', data=feature_importance)

plt.title("Feature Importance")
plt.show()

---

## Section 5 - Export for BI Layer

The cleaned frame is exported for Power BI and Excel. Everything from here is handoff:
the analysis is done, and these cells exist so the dashboard consumes the *same* cleaned
population the notebook analysed - not a separately-cleaned copy that could drift.

In [ ]:
# ==========================================
# EXPORT FOR THE BI LAYER
# ==========================================

df.to_csv("data/Uber_Cleaned_data.csv", index=False)
print("Exported to data/Uber_Cleaned_data.csv")


In [ ]:
# Excel report for stakeholders - built from the CLEANED frame `df`, not a
# fresh read of the raw CSV, so the workbook matches the analysis above.

summary = df.pivot_table(index='Vehicle_Type',
                         values='Fare_Amount',
                         aggfunc='mean')

with pd.ExcelWriter('excel_analysis/Uber_Report.xlsx') as writer:
    df.to_excel(writer, sheet_name='Cleaned_Data', index=False)
    summary.to_excel(writer, sheet_name='Summary')

print("Excel Report Created!")